In [11]:
import pandas as pd
from dotenv import load_dotenv
from supabase import Client, create_client
import os

In [2]:
data = pd.read_csv(
    "./data/global_wordfreq.release_UTF-8.txt", sep="\t", header=None
).rename(
    columns={0:"char", 1:"freq"}
)

In [3]:
data

,char,freq
0,第,2002074595
1,的,943370349
2,了,255733044
3,在,197672850
4,是,171296602
...,...,...
1048571,赤崖堡村,25
1048572,经军,25
1048573,于松如,25
1048574,11．86％,25


In [6]:
# Calculate relative frequency as percentage
total_instances = sum(data.freq)
data["rel_freq_pct"] = data.freq / total_instances * 100
data.head()

,char,freq,rel_freq_pct
0,第,2002074595,10.632977
1,的,943370349,5.010220
2,了,255733044,1.358193
3,在,197672850,1.049836
4,是,171296602,0.909753


In [12]:
# Grab data from our database and compare
def get_all_vocabulary(client: Client):
    range_start = 0
    all_data = []

    while True:
        response = client.table("vocabulary").select("*").range(range_start, range_start + 999).execute()
        data = response.data
        all_data += data

        if len(data) < 1000:
            break

        range_start += 1000
    
    return all_data

load_dotenv()
client = create_client(
    os.environ["DB_URL"],
    os.environ["DB_SERVICE_KEY"]
)

vocabulary_data = get_all_vocabulary(client)

In [17]:
# Get frequency of matches with both traditional and simplified
trad_char_set = set([entry["traditional"] for entry in vocabulary_data])
simp_char_set = set([entry["simplified"] for entry in vocabulary_data])

# Get set of characters in dataframe
df_char_set = set(data.char)

In [23]:
trad_char_intersect = len(trad_char_set.intersection(df_char_set))
simp_char_intersect = len(simp_char_set.intersection(df_char_set))

print(f"Traditional overlap: {trad_char_intersect}")
print(f"Simplified overlap: {simp_char_intersect}")

Traditional overlap: 3594
Simplified overlap: 6937


In [39]:
# Update original data and upsert back into table
df_filtered = data[data["char"].isin(simp_char_set)]

# Convert to dict for quick look-up
freq_lookup = dict()
for k, v in df_filtered.iterrows():
    char = v["char"]
    freq = v["rel_freq_pct"]
    freq_lookup[char] = freq

freq_lookup

{'第': 10.632976602799742,
 '的': 5.010220335317738,
 '了': 1.358192886621568,
 '在': 1.049836401854319,
 '是': 0.9097526964049505,
 '我': 0.8996332521664626,
 '和': 0.6313608945721991,
 '有': 0.5006743667401303,
 '你': 0.4699563638888518,
 '个': 0.4666213557215754,
 '也': 0.41032226722145654,
 '这': 0.389044561827894,
 '不': 0.37258318168617266,
 '他': 0.3561069201806323,
 '上': 0.353232830657324,
 '人': 0.34790258849971945,
 '中': 0.34765722656999937,
 '就': 0.3452811847464976,
 '年': 0.31775765624093405,
 '为': 0.3049529490348917,
 '对': 0.3003197888349387,
 '说': 0.2813054209745126,
 '都': 0.2789724405706556,
 '要': 0.26954830973373384,
 '到': 0.2689775706611254,
 '着': 0.23975779505229883,
 '与': 0.2310161250471922,
 '将': 0.2286769414195877,
 '日': 0.22444658181782437,
 '我们': 0.2219331077846348,
 '好': 0.22055609833226347,
 '月': 0.21765124761722077,
 '会': 0.2125818382718505,
 '大': 0.20373964205176648,
 '来': 0.20165805685004165,
 '还': 0.1985494875934677,
 '等': 0.19259412193587888,
 '而': 0.1900681511685472,
 '地

In [40]:
# Add to dataset
count_match = 0
for entry in vocabulary_data:
    simp_char = entry["simplified"]
    freq = freq_lookup.get(simp_char, None)

    if freq: count_match += 1

    entry["relative_freq_pct"] = freq

print(count_match)

7420


In [41]:
# Upsert
repsonse = client.table("vocabulary").upsert(vocabulary_data).execute()

In [43]:
# Query to validate
response = client.table("vocabulary").select("*").is_("relative_freq_pct", "null").execute()

None


In [45]:
len(response.data)

373